# C4 SE path simulation v1 (`se-path-sim-v1`)

S1 scope: **46** C1 practice topics with semester eligibility masks.
Additive package `training_se_v1` — does **not** modify `training_se` (v0).

Synthetic simulation only. Not real student behaviour. No automatic wiring of supervised response models into this env.

Run cells **in order**. `RUN_FULL_TRAINING` and `RUN_FINAL_EVALUATION` default **False**.
Artifacts go under a **new** Drive folder (`se_path_sim_v1`), never into v0 paths.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Dependencies


In [2]:
%pip install -q "numpy>=1.23,<3" "gymnasium>=0.29.1,<2.0" "stable-baselines3>=2.3.0,<3.0" "sb3-contrib>=2.3.0,<3.0" "torch>=2.0" "pandas>=1.5,<3" cloudpickle tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 6.0 MB/s eta 0:00:00


## 2. Upload & extract `c4-se-v1-source-colab.zip`


In [3]:
from pathlib import Path
import zipfile, shutil
from google.colab import files

UPLOAD_DIR = Path('/content/c4_se_v1_upload')
EXTRACT_DIR = Path('/content/c4_se_v1_src')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print('Upload c4-se-v1-source-colab.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded')
zip_name = next(iter(uploaded))
zip_path = UPLOAD_DIR / zip_name
zip_path.write_bytes(uploaded[zip_name])
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)
print('Top:', list(EXTRACT_DIR.iterdir())[:10])


Upload c4-se-v1-source-colab.zip


Saving c4-se-v1-source-colab.zip to c4-se-v1-source-colab.zip
Extracted to /content/c4_se_v1_src
Top: [PosixPath('/content/c4_se_v1_src/ai-services')]


## 3. Drive mount (v1 output root only)


In [4]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts root:', DRIVE_ROOT)
print('Do not write into se_path_sim_v0')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifacts root: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1
Do not write into se_path_sim_v0


## 4. Imports / versions / paths


In [5]:
import sys, os, json
from pathlib import Path

if 'EXTRACT_DIR' not in globals() or 'DRIVE_ROOT' not in globals():
    raise RuntimeError('Run cells 2 (extract) and 3 (Drive mount) first.')

hits = list(Path(EXTRACT_DIR).rglob('training_se_v1/protocol.py'))
if not hits:
    raise FileNotFoundError('training_se_v1/protocol.py not found; re-upload ZIP')
C4_ROOT = hits[0].parents[1].resolve()
sys.path.insert(0, str(C4_ROOT))
os.chdir(C4_ROOT)
print('C4_ROOT', C4_ROOT)

from training_se_v1.colab_bootstrap import require_symbol
import gymnasium, numpy, torch, stable_baselines3, sb3_contrib
from training_se_v1.protocol import PROTOCOL_VERSION, GENERATOR_VERSION
print('protocol', PROTOCOL_VERSION, GENERATOR_VERSION)
print('gymnasium', gymnasium.__version__, 'sb3', stable_baselines3.__version__, 'sb3_contrib', sb3_contrib.__version__, 'torch', torch.__version__)


C4_ROOT /content/c4_se_v1_src/ai-services/c4
protocol se-path-sim-v1 se-sim-1.0.0
gymnasium 1.3.0 sb3 2.9.0 sb3_contrib 2.9.0 torch 2.11.0+cpu


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Frozen catalogue / observation schema / assumptions


In [6]:
from training_se_v1.catalogue import (
    catalogue_payload, catalogue_hash, ORDERED_TOPICS, N_TOPICS, validate_catalogue,
)
from training_se_v1.config import frozen_config_dict, ASSUMPTIONS
from training_se_v1.observation import observation_schema, OBS_DIM

print(validate_catalogue())
print('n_topics', N_TOPICS, 'obs_dim', OBS_DIM)
print('catalogue_sha256', catalogue_hash())
print('first/last topic', ORDERED_TOPICS[0], ORDERED_TOPICS[-1])
print(json.dumps(observation_schema(), indent=2)[:1200])
print(json.dumps(ASSUMPTIONS, indent=2))
print(json.dumps(frozen_config_dict(), indent=2)[:1500])
(DRIVE_ROOT / '_session_catalogue_preview.json').write_text(
    json.dumps({**catalogue_payload(), 'sha256': catalogue_hash()}, indent=2), encoding='utf-8'
)


{'ok': True, 'n_topics': 46, 'catalogue_hash': '95aaba520a28a494e9d6fed0c30de5999b6d47f376368424fb211acb434a1622', 'y3s1': 28, 'y3s2': 18}
n_topics 46 obs_dim 96
catalogue_sha256 95aaba520a28a494e9d6fed0c30de5999b6d47f376368424fb211acb434a1622
first/last topic interaction_functional_design af_framework_selection
{
  "protocol": "se-path-sim-v1",
  "dim": 96,
  "dtype": "float32",
  "range": [
    0.0,
    1.0
  ],
  "order": [
    "mastery::interaction_functional_design",
    "mastery::code_quality_solid",
    "mastery::refactoring_best_practices",
    "mastery::unit_integration_testing",
    "mastery::debugging_profiling_config",
    "mastery::project_integration_scope",
    "mastery::schedule_cost_management",
    "mastery::project_quality_management",
    "mastery::communication_stakeholders",
    "mastery::project_risk_management",
    "mastery::ux_foundations_process",
    "mastery::user_research_synthesis",
    "mastery::structuring_prototyping",
    "mastery::visual_design_acces

11704

## 6. Environment + semester-mask + reproducibility checks


In [7]:
import numpy as np
from training_se_v1.environment import make_env
from training_se_v1.config import SeEnvConfig
from catalogues.c1_practice_full_2026_10 import Y3S1_INDICES, Y3S2_INDICES

cfg = SeEnvConfig()
e1 = make_env(cfg, scenario='y3s1_sim_val', seed=0)
e2 = make_env(cfg, scenario='y3s1_sim_val', seed=0)
o1, i1 = e1.reset(seed=123)
o2, i2 = e2.reset(seed=123)
assert np.allclose(o1, o2)
assert o1.shape == (OBS_DIM,)
assert i1['synthetic'] is True
assert not any(bool(i1['action_mask'][i]) for i in Y3S2_INDICES)
print('y3s1 repro OK; mask sum', int(i1['action_mask'].sum()))
e1.close(); e2.close()

e3 = make_env(cfg, scenario='y3s2_sim_val', seed=1)
_, i3 = e3.reset(seed=1)
assert not any(bool(i3['action_mask'][i]) for i in Y3S1_INDICES)
print('y3s2 mask OK; mask sum', int(i3['action_mask'].sum()))
e3.close()


y3s1 repro OK; mask sum 22
y3s2 mask OK; mask sum 14


## 7. Smoke training + validation selection (both semester scenarios)


In [8]:
from training_se_v1.pipeline import run_training_and_validation

smoke = run_training_and_validation(
    DRIVE_ROOT,
    mode='smoke',
    timesteps=2000,
    seeds=[0],
    eval_episodes=8,
    device='cpu',
    cfg=cfg,
    run_full=False,
)
SMOKE_RUN_DIR = Path(smoke['run_dir'])
print('smoke run', SMOKE_RUN_DIR)
print('selected', smoke['selection'].get('selected'))
print(json.dumps(smoke['selection'], indent=2)[:2500])


smoke run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1/train_smoke_se-path-sim-v1_20261013T180625.114Z
selected maskable_ppo
{
  "domain": "se_simulation",
  "synthetic": true,
  "protocol": "se-path-sim-v1",
  "generator": "se-sim-1.0.0",
  "not_real_se_students": true,
  "not_assistments": true,
  "comparison_labels": {
    "maskable_ppo": "MaskablePPO (sb3-contrib; masks in training and evaluation)",
    "weakest_stored_mastery_v1_sim": "Weakest mastery among masked-valid topics using simulator obs mastery (compatible ranking rule with Node weakest_stored_mastery_v1; state source differs \u2014 synthetic obs, not C1 insights)",
    "random_valid": "Uniform random among masked-valid actions (separate RNG stream)",
    "recent_performance": "Lowest rolling in-episode attempt accuracy; cold-start = lowest-index untried valid"
  },
  "env_protocol_version": "se-path-sim-v1",
  "note": "Selection uses mean of Y3S1+Y3S2 validation returns only; do not tune on final test. 

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. Schema-aware comparison table (returns / mastery — not accuracy)


In [9]:
import pandas as pd

require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
sel = json.loads((SMOKE_RUN_DIR / 'selection_val.json').read_text())
rows = []
by_alg = sel.get('mean_val_return_by_algorithm') or sel.get('by_algorithm') or {}
for method, stats in by_alg.items():
    if isinstance(stats, dict):
        rows.append({
            'method': method,
            'val_return_mean': stats.get('mean'),
            'val_return_sample_sd': stats.get('sample_sd'),
            'selected': method == sel.get('selected'),
            'domain': 'se_simulation',
            'synthetic': True,
            'protocol': 'se-path-sim-v1',
        })
df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values('val_return_mean', ascending=False)
display(df)
print('Label: simulation_only — RL metrics are returns/mastery/progress, not classification accuracy')


""


Label: simulation_only — RL metrics are returns/mastery/progress, not classification accuracy


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_sim_v1/"
    "train_full_se-path-sim-v1_20261013T181643.681Z"
)

assert RUN_DIR.is_dir(), f"Run folder not found: {RUN_DIR}"

def find_selection(obj):
    if isinstance(obj, dict):
        semesters = obj.get("by_semester")
        if isinstance(semesters, dict) and any(
            isinstance(v, dict)
            and "mean_val_return_by_algorithm" in v
            for v in semesters.values()
        ):
            return obj
        for value in obj.values():
            found = find_selection(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_selection(value)
            if found is not None:
                return found
    return None

selection = None
for path in sorted(RUN_DIR.rglob("*.json")):
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except (ValueError, UnicodeError):
        continue
    selection = find_selection(obj)
    if selection is not None:
        print("Source:", path)
        break

if selection is None:
    raise RuntimeError(
        "Saved semester summary not found. "
        "Share the notebook's smoke/summary cell code."
    )

rows = []
invalid_rows = []

for semester, data in selection["by_semester"].items():
    for method, metrics in data.get(
        "mean_val_return_by_algorithm", {}
    ).items():
        rows.append({
            "semester": semester,
            "method": method,
            "validation_return": metrics["mean"],
            "reported_seed_sd": metrics.get("sample_sd"),
        })

    for seed, methods in data.get("seeds", {}).items():
        for method, result in methods.items():
            invalid = result.get("summary", {}).get(
                "invalid_action_rate", {}
            )
            invalid_rows.append({
                "semester": semester,
                "seed": seed,
                "method": method,
                "invalid_action_rate": invalid.get("mean"),
            })

display(pd.DataFrame(rows))
display(pd.DataFrame(invalid_rows))
print("Selected:", selection.get("selected", "See notebook selection output"))

Source: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1/train_full_se-path-sim-v1_20261013T181643.681Z/comparison_val.json


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,semester,method,validation_return,reported_seed_sd
0,Y3S1,maskable_ppo,11.138666,0.296765
1,Y3S1,random_valid,9.010667,0.168776
2,Y3S1,recent_performance,8.702667,0.299395
3,Y3S1,weakest_stored_mastery_v1_sim,8.460000,0.218687
4,Y3S2,maskable_ppo,10.914666,0.182179
5,Y3S2,random_valid,9.514667,0.250961
6,Y3S2,recent_performance,9.356000,0.148162
7,Y3S2,weakest_stored_mastery_v1_sim,8.945334,0.224583


,semester,seed,method,invalid_action_rate
0,Y3S1,0,maskable_ppo,0.0
1,Y3S1,0,random_valid,0.0
2,Y3S1,0,recent_performance,0.0
3,Y3S1,0,weakest_stored_mastery_v1_sim,0.0
4,Y3S1,1,maskable_ppo,0.0
5,Y3S1,1,random_valid,0.0
6,Y3S1,1,recent_performance,0.0
7,Y3S1,1,weakest_stored_mastery_v1_sim,0.0
8,Y3S1,2,maskable_ppo,0.0
9,Y3S1,2,random_valid,0.0


Selected: maskable_ppo


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Full training (gated)


In [11]:
# Set True only for a multi-seed full run. Creates a NEW run folder under se_path_sim_v1.
RUN_FULL_TRAINING = True
FULL_TIMESTEPS = 50_000
FULL_SEEDS = [0, 1, 2]

FULL_RUN_DIR = None
if RUN_FULL_TRAINING:
    full = run_training_and_validation(
        DRIVE_ROOT,
        mode='full',
        timesteps=FULL_TIMESTEPS,
        seeds=FULL_SEEDS,
        eval_episodes=20,
        device='cuda',
        cfg=cfg,
        run_full=True,
    )
    FULL_RUN_DIR = Path(full['run_dir'])
    print('full run', FULL_RUN_DIR)
else:
    print('Skipped full training (RUN_FULL_TRAINING=False)')


full run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1/train_full_se-path-sim-v1_20261013T181643.681Z


## 10. Final simulator evaluation (gated; after frozen selection)


In [13]:
from training_se_v1.pipeline import run_final_evaluation

RUN_FINAL_EVALUATION = True
FINAL_TARGET_RUN = FULL_RUN_DIR or globals().get('SMOKE_RUN_DIR')

if RUN_FINAL_EVALUATION:
    require_symbol('FINAL_TARGET_RUN', FINAL_TARGET_RUN)
    plan = run_final_evaluation(FINAL_TARGET_RUN, run_final=True, eval_episodes=20, cfg=cfg)
    print(plan)
else:
    print('Skipped final evaluation (RUN_FINAL_EVALUATION=False); '
          'validation selection remains frozen separately from final test.')


{'domain': 'se_simulation', 'synthetic': True, 'protocol': 'se-path-sim-v1', 'generator': 'se-sim-1.0.0', 'not_real_se_students': True, 'not_assistments': True, 'selection_frozen_from': '/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1/train_full_se-path-sim-v1_20261013T181643.681Z/selection_val.json', 'selected_on_validation': 'maskable_ppo', 'train_seeds': [0, 1, 2], 'n_episodes': 20, 'test_scenarios': ['y3s1_sim_test', 'y3s2_sim_test'], 'note': 'Final simulator test reporting only; not real SE students.'}


In [15]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_sim_v1/"
    "train_full_se-path-sim-v1_20261013T181643.681Z"
)

assert RUN_DIR.is_dir(), f"Run folder not found: {RUN_DIR}"

# Find saved final-test CSV summaries.
csv_files = sorted(
    p for p in RUN_DIR.rglob("*.csv")
    if "test" in str(p.relative_to(RUN_DIR)).lower()
    and any(word in p.name.lower()
            for word in ("summary", "comparison", "metrics"))
)

if csv_files:
    for path in csv_files:
        print("\nResults:", path.relative_to(RUN_DIR))
        table = pd.read_csv(path)
        with pd.option_context(
            "display.max_columns", None,
            "display.max_rows", 100,
            "display.max_colwidth", 80,
        ):
            display(table)
else:
    print("Final-test CSV summary not found.")
    print("Available test-result files:")
    result_files = sorted(
        p for p in RUN_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() in (".json", ".csv")
        and "test" in str(p.relative_to(RUN_DIR)).lower()
    )
    for path in result_files:
        print(path.relative_to(RUN_DIR))

    if not result_files:
        print("No saved test-result JSON/CSV files found.")


Results: export_bundle/final_simulator_test/test_summary.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,semester,method,return_mean,return_sample_sd
0,Y3S1,maskable_ppo,11.157333,0.085541
1,Y3S1,weakest_stored_mastery_v1_sim,8.712000,0.252000
2,Y3S1,random_valid,9.710667,0.042771
3,Y3S1,recent_performance,9.262667,0.198649
4,Y3S2,maskable_ppo,11.082666,0.202557
5,Y3S2,weakest_stored_mastery_v1_sim,9.477333,0.382211
6,Y3S2,random_valid,10.140000,0.270022
7,Y3S2,recent_performance,10.084000,0.211395



Results: final_simulator_test/test_summary.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,semester,method,return_mean,return_sample_sd
0,Y3S1,maskable_ppo,11.157333,0.085541
1,Y3S1,weakest_stored_mastery_v1_sim,8.712000,0.252000
2,Y3S1,random_valid,9.710667,0.042771
3,Y3S1,recent_performance,9.262667,0.198649
4,Y3S2,maskable_ppo,11.082666,0.202557
5,Y3S2,weakest_stored_mastery_v1_sim,9.477333,0.382211
6,Y3S2,random_valid,10.140000,0.270022
7,Y3S2,recent_performance,10.084000,0.211395


## 11. Artifact export reminder


In [14]:
require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
exp = SMOKE_RUN_DIR / 'export_bundle'
print('Export bundle:', exp)
print('Files:', sorted(p.name for p in exp.iterdir()) if exp.is_dir() else 'missing')
print('Label: simulation_only / domain=se_simulation / protocol=se-path-sim-v1')
print('Manifest:', (exp / 'manifest.json').is_file())
print('Student Adaptive path remains heuristic; no model wiring from this notebook.')


Export bundle: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v1/train_smoke_se-path-sim-v1_20261013T180625.114Z/export_bundle
Files: ['MODEL_CARD_SE_SIM.md', 'comparison_val.json', 'manifest.json', 'models', 'observation_schema.json', 'protocol.json', 'selection_val.json', 'synthetic_provenance.json', 'topic_catalogue.json', 'train_config.json']
Label: simulation_only / domain=se_simulation / protocol=se-path-sim-v1
Manifest: True
Student Adaptive path remains heuristic; no model wiring from this notebook.
